In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import spikeinterface as si
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages

from tqdm import tqdm


import sys
import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre


import torch.nn.functional as F
from pathlib import Path


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import time
import pickle
import networkx as nx
from scipy.spatial import ConvexHull
# from function.Function import *

In [3]:
def count_array2_in_range_of_array1(array1, array2, threshold=5):

    sorted_array1 = np.sort(array1)
    array2 = np.sort(array2)
    
    lefts = array2 - threshold
    rights = array2 + threshold
    
    left_indices = np.searchsorted(sorted_array1, lefts, side='left')
    
    right_indices = np.searchsorted(sorted_array1, rights, side='right')
    
    has_within_range = right_indices > left_indices
    
    count = np.sum(has_within_range)
    
    return count

def label_array1_based_on_array2(array1, array2, threshold=5):
    array_1 = np.sort(array1)
    sorted_array2 = np.sort(array2)
    
    labels = np.zeros(len(array1), dtype=int)
    
    for i, value in enumerate(array1):
        left = value - threshold
        right = value + threshold
        
        left_index = np.searchsorted(sorted_array2, left, side='left')
        right_index = np.searchsorted(sorted_array2, right, side='right')
        
        if right_index > left_index:
            labels[i] = 1
    
    return labels
def detect_local_minimum_in_window(data, window_size=20, std_multiplier=2):

    """
    在每个滑动窗口范围内检测局部最小值的索引，并确保最小值低于 mean - std_multiplier * std。

    参数:
    data : numpy.ndarray
        输入数据，形状为 (n_rows, n_columns)。
    window_size : int
        滑动窗口的大小，用于定义局部范围，默认为 20。
    std_multiplier : float
        标准差的倍数，用于筛选局部最小值，默认为 2。

    返回:
    local_minima_indices : list of numpy.ndarray
        每行局部最小值的索引列表，每个元素是对应行局部最小值的索引数组。
    """
    local_minima_indices = []

    for row in data:
        minima_indices = []
        row = row.astype(np.float32)
        row_mean = np.mean(row)
        row_std = np.std(row)
        threshold = row_mean - std_multiplier * row_std

        for start in range(0, len(row), window_size):
            end = min(start + window_size, len(row))
            window = row[start:end]
            
            if len(window) > 0:
                local_min_index = np.argmin(window)
                local_min_value = window[local_min_index]
                
                if local_min_value < threshold:
                    minima_indices.append(start + local_min_index)  
        
        local_minima_indices.extend(minima_indices)
        local_minima_indices = list(set(local_minima_indices))  

    return local_minima_indices


def cluster_label_array1_based_on_array2(array1, array2, threshold=5):

    """
    根据 array2 的 'time' 和 'cluster' 对 array1 进行标记。
    如果 array1 中的某个值在 threshold 范围内存在于 array2 的 'time' 中，则标记为对应的 'cluster' 值，否则为 0。
    
    参数:
    array1 : numpy.ndarray
        要标记的数组。
    array2 : numpy.ndarray
        包含 'time' 和 'cluster' 的二维数组。
        第一列为 'time'，第二列为 'cluster'。
    threshold : int
        判断范围的阈值。
    
    返回:
    labels : numpy.ndarray
        长度为 len(array1) 的标签数组，值为 array2 中的 'cluster' 或 0。
    """

    array2 = np.array(array2.iloc[:, [5, 1]])
    sorted_indices = np.argsort(array2[:, 0])
    sorted_array2 = array2[sorted_indices]
    
    labels = np.zeros(len(array1), dtype=int)
    
    # 遍历 array1 中的每个元素
    for i, value in enumerate(array1):
        # 计算当前值的范围
        left = value - threshold
        right = value + threshold
        
        left_index = np.searchsorted(sorted_array2[:, 0], left, side='left')
        right_index = np.searchsorted(sorted_array2[:, 0], right, side='right')
        
        # 如果范围内存在值，则标记为对应的 'cluster'
        if right_index > left_index:
            # 获取范围内的第一个匹配值的 'cluster'
            labels[i] = sorted_array2[left_index, 1]
    
    return labels


def label_array1_based_on_array2(array1, array2, threshold=5):

    """
    根据 array2 的值对 array1 进行标记。
    如果 array1 中的某个值在 threshold 范围内存在于 array2 中，则标记为 1，否则为 0。
    
    参数:
    array1 : numpy.ndarray
        要标记的数组。
    array2 : numpy.ndarray
        用于判断的数组。
    threshold : int
        判断范围的阈值。
    
    返回:
    labels : numpy.ndarray
        长度为 len(array1) 的标签数组，值为 0 或 1。
    """
    # 对 array2 进行排序以加速搜索
    sorted_array2 = np.sort(array2)
    
    # 初始化标签数组，默认值为 0
    labels = np.zeros(len(array1), dtype=int)
    
    # 遍历 array1 中的每个元素
    for i, value in enumerate(array1):
        # 计算当前值的范围
        left = value - threshold
        right = value + threshold
        
        # 使用二分搜索判断范围内是否存在值
        left_index = np.searchsorted(sorted_array2, left, side='left')
        right_index = np.searchsorted(sorted_array2, right, side='right')
        
        # 如果范围内存在值，则标记为 1
        if right_index > left_index:
            labels[i] = 1
    
    return labels


def extract_windows(data, indices, window_size=61):
    """
    根据给定的时间点索引提取窗口。
    
    参数:
    data : numpy.ndarray
        输入数据，形状为 (n_channels, time)
    indices : numpy.ndarray
        时间点索引数组，用于指定需要提取窗口的中心点
    window_size : int
        窗口长度，默认为61（对应time-30到time+31）
    
    返回:
    windows : numpy.ndarray
        提取的窗口数据，形状为 (len(indices), n_channels, window_size)
    """
    n_channels, time_length = data.shape
    half_window = window_size // 2

    if np.any(indices < half_window) or np.any(indices >= time_length - half_window):
        raise ValueError("Some indices are out of bounds for the given window size.")

    windows = []
    for idx in indices:
        window = data[:, idx - half_window:idx + half_window + 1]
        windows.append(window)

    windows = np.array(windows)
    return windows

In [28]:
recording_raw = se.MEArecRecordingExtractor(file_path='/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_type1.h5')
probe_384channel = recording_raw.get_probegroup()

In [29]:
recording_raw = se.read_binary(file_paths='/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/raw_data/810755807/spike_band.dat', sampling_frequency=30000, dtype=np.int16, num_channels=384)
recording_raw = recording_raw.set_probegroup(probe_384channel)
recording_f = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_f = spre.common_reference(recording_f, reference="global", operator="median")
recording_f = recording_f.time_slice(start_time=0, end_time= 1200)

recording_f = spre.resample(recording_f, 10000)


In [20]:
def build_probe_group():
    print("[INFO] Loading probe template")
    template_recording = se.MEArecRecordingExtractor(file_path=str('/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_type1.h5'))
    probegroup = template_recording.get_probegroup()
    offset = 0
    for probe in probegroup.probes:
        n_contacts = probe.get_contact_count()
        device_indices = np.arange(offset, offset + n_contacts, dtype=int)
        probe.set_device_channel_indices(device_indices)
        offset += n_contacts
    return probegroup
from typing import Dict, Iterable, List, Sequence, Tuple

from dataclasses import dataclass
@dataclass
class CliqueInfo:
    clique_id: int
    device_channel_indices: List[int]
    contact_ids: List[str]
    center: Tuple[float, float]

In [21]:
def build_sliding_cliques(
    probe_group,
    clique_size: int = 50,
    min_size: int = 25,
    min_overlap: int = 16,
    target_groups: int = 11,
)-> List[CliqueInfo]:
    df = probe_group.to_dataframe()
    if "device_channel_indices" in df.columns:
        device_indices = df["device_channel_indices"].astype(int).to_numpy()
    else:
        device_indices = np.arange(len(df), dtype=int)
    positions = df.loc[:, ["x", "y"]].to_numpy()
    contact_ids = df["contact_ids"].astype(str).to_numpy()

    order = np.argsort(positions[:, 1])
    ordered_device = device_indices[order]
    ordered_contacts = contact_ids[order]
    ordered_positions = positions[order]

    step = clique_size - min_overlap
    cliques: List[CliqueInfo] = []

    start_indices = list(range(0, len(ordered_device) - clique_size + 1, step))
    if start_indices[-1] + clique_size < len(ordered_device):
        start_indices.append(len(ordered_device) - clique_size)

    for idx, start in enumerate(start_indices[:target_groups]):
        slice_device = ordered_device[start : start + clique_size]
        slice_positions = ordered_positions[start : start + clique_size]
        slice_contacts = ordered_contacts[start : start + clique_size]
        if len(slice_device) < min_size:
            continue
        center = tuple(np.mean(slice_positions, axis=0))
        cliques.append(
            CliqueInfo(
                clique_id=idx,
                device_channel_indices=list(slice_device),
                contact_ids=list(slice_contacts),
                center=center,
            )
        )

    print(f"[INFO] Built {len(cliques)} cliques (target {target_groups})")
    for info in cliques:
        print(
            f"       Clique {info.clique_id:02d}: channels {info.device_channel_indices[0]}-"
            f"{info.device_channel_indices[-1]} ({len(info.device_channel_indices)} channels)"
        )

    return cliques

In [22]:
probe_group = build_probe_group()
cliques = build_sliding_cliques(
    probe_group,
    clique_size=50,
    min_size=25,
    min_overlap=16,
    target_groups=11,
)

[INFO] Loading probe template
[INFO] Built 11 cliques (target 11)
       Clique 00: channels 192-204 (50 channels)
       Clique 01: channels 104-308 (50 channels)
       Clique 02: channels 209-29 (50 channels)
       Clique 03: channels 121-133 (50 channels)
       Clique 04: channels 34-46 (50 channels)
       Clique 05: channels 330-342 (50 channels)
       Clique 06: channels 243-255 (50 channels)
       Clique 07: channels 155-167 (50 channels)
       Clique 08: channels 260-80 (50 channels)
       Clique 09: channels 172-376 (50 channels)
       Clique 10: channels 179-383 (50 channels)


In [23]:
spike_inf = pd.read_csv("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_sorting/810755807/spike_inf.tsv", index_col=0, sep='\t')
cluster_inf = pd.read_csv("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_sorting/810755807/cluster_inf.csv", index_col=0)

In [24]:
class CustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]
    

class Spike_Detection_MLP(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size, n_channels, time_window):
        super(Spike_Detection_MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size2, 16)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(16, output_size)
        self.sigmoid = nn.Sigmoid()  

        self.n_channels = n_channels
        self.time_window = time_window
    def forward(self, x):
        x = x.reshape(-1, self.n_channels * self.time_window)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.fc4(x)
        x = self.sigmoid(x)
        return x

In [46]:
probe_df = probe_group.to_dataframe()


In [ ]:
probe_df

In [ ]:
accuracy_dict = {}
tpr_dict = {}
tnr_dict = {}
probe_df = probe_group.to_dataframe()
for clique_id, clique in enumerate(cliques):
    print(f'Processing {clique_id}...')
    os.makedirs(f'/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_detection/train_result/810755807/{clique_id}', exist_ok=True)
    accuracy_dict[clique_id] = []
    tpr_dict[clique_id] = []
    tnr_dict[clique_id] = []

    total_frames = int(recording_f.get_total_duration() * 10000)
    chunk_size = 120000  
    window_size = 31
    half_window = window_size // 2

    all_valid_indices = []
    all_windows = []
    for start_frame in tqdm(range(0, total_frames, chunk_size)):
        end_frame = min(start_frame + chunk_size, total_frames)
        
        data_chunk = recording_f.get_traces(
            start_frame=start_frame,
            end_frame=end_frame,
            channel_ids=clique.device_channel_indices
        )  # shape: (n_channels, chunk_size)
        
        threshold_result = detect_local_minimum_in_window(
            data_chunk.T,  
            std_multiplier=1.5,
            window_size= 20
        )
        
        threshold_result = np.array(threshold_result) + start_frame
        valid_indices = threshold_result[
            (threshold_result >= start_frame + half_window + 1) & 
            (threshold_result < end_frame - half_window)
        ]
        
        for idx in valid_indices:
            rel_idx = idx - start_frame
            window = data_chunk.T[:, rel_idx-half_window : rel_idx+half_window+1]
            all_windows.append(window)
        
        all_valid_indices.extend(valid_indices)

    all_valid_indices = np.array(all_valid_indices)
    all_windows = np.stack(all_windows)  

    channel_mask = probe_df["device_channel_indices"].isin(clique.device_channel_indices)
    clique_positions = probe_df.loc[channel_mask, ["x", "y"]].to_numpy()

    if clique_positions.size == 0:
        y_min = float(cluster_inf["position_2"].min())
        y_max = float(cluster_inf["position_2"].max())
    else:
        if clique_positions.shape[0] >= 3:
            hull = ConvexHull(clique_positions)
            hull_points = clique_positions[hull.vertices]
        else:
            hull_points = clique_positions
        y_min = float(hull_points[:, 1].min())
        y_max = float(hull_points[:, 1].max())

    margin = 60.0
    if y_max - y_min > 2 * margin:
        y_lower = y_min + margin
        y_upper = y_max - margin
    else:
        y_lower = y_min
        y_upper = y_max

    cluster_y = cluster_inf["position_2"].to_numpy()
    cluster_ids = cluster_inf["cluster_id"].to_numpy()
    mask = (cluster_y >= y_lower) & (cluster_y <= y_upper)
    indices_within_range = cluster_ids[mask]
    spike_inf_temp = spike_inf[spike_inf['cluster'].isin(indices_within_range)]

    labels = label_array1_based_on_array2(all_valid_indices, spike_inf_temp['time'], threshold=1)
    indices_0 = np.where(labels == 0)[0] 
    indices_1 = np.where(labels == 1)[0] 
    #print(len(indices_1) / len(spike_inf_temp))
    target_0_count = len(indices_1) 

    if len(indices_0) > target_0_count:
        sampled_indices_0 = np.random.choice(indices_0, target_0_count, replace=False)
    else:
        sampled_indices_0 = indices_0  

    final_indices = np.concatenate([sampled_indices_0, indices_1])

    np.random.shuffle(final_indices)

    sampled_windows = all_windows[final_indices]
    sampled_labels = labels[final_indices]

    dataset = CustomDataset(sampled_windows, sampled_labels)

    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

    batch_size = 1024 
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    hidden_size1 = 128
    hidden_size2 = 32
    output_size = 1  
    device = 'cuda'
    input_size = sampled_windows.shape[1] * sampled_windows.shape[2]


    criterion = nn.BCELoss()  

    for trail in range(1, 6):
        print(f"Training for trail {trail}...")

        model = Spike_Detection_MLP(input_size, hidden_size1, hidden_size2, 
                                    output_size, n_channels=sampled_windows.shape[1], time_window= sampled_windows.shape[2])
        model = model.to(device)

        optimizer = optim.Adam(model.parameters(), lr=0.0001)

        num_epochs = 50
        tpr_best = 0
        i = 0
        for epoch in range(num_epochs):
            model.train()
            total_loss = 0
            correct = 0
            total = 0
            for batch_data, batch_labels in train_loader:
                batch_labels = batch_labels.float().unsqueeze(1)

                batch_data = batch_data.to(device)
                batch_labels = batch_labels.to(device)

                outputs = model(batch_data)
                loss = criterion(outputs, batch_labels)

                predicted = (outputs > 0.5).float()  
                total += batch_labels.size(0)
                correct += (predicted == batch_labels).sum().item()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            #print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")
            #print(f"Train Accuracy: {100 * correct / total:.2f}%")

            model.eval()
            correct = 0
            total = 0

            true_positive = 0
            true_negative = 0
            false_positive = 0
            false_negative = 0

            with torch.no_grad():
                for batch_data, batch_labels in test_loader:
                    batch_labels = batch_labels.float().unsqueeze(1)
                    batch_data = batch_data.to(device)
                    batch_labels = batch_labels.to(device)

                    outputs = model(batch_data)
                    predicted = (outputs > 0.5).float()  
                    total += batch_labels.size(0)
                    correct += (predicted == batch_labels).sum().item()
                    true_positive += ((predicted == 1) & (batch_labels == 1)).sum().item()
                    true_negative += ((predicted == 0) & (batch_labels == 0)).sum().item()
                    false_positive += ((predicted == 1) & (batch_labels == 0)).sum().item()
                    false_negative += ((predicted == 0) & (batch_labels == 1)).sum().item()

            tpr = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
            tnr = true_negative / (true_negative + false_positive) if (true_negative + false_positive) > 0 else 0

            #print(f"Test Accuracy: {100 * correct / total:.2f}%")
            #print(f"Test TPR: {100 * tpr:.2f}%")
            #print(f"Test TNR: {100 * tnr:.2f}%")

            if tpr > tpr_best:
                        tpr_best = tpr
                        i = 0
                        torch.save(model, f'/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_detection/train_result/810755807/{clique_id}/trail_{trail}.pth')
                        #print(f"Best model saved with TPR: {tpr_best:.4f}")
                        #print("_" * 60)

            else:
                i += 1
                if i == 3:
                    #print(f"Training stopped after {epoch+1} epochs with best TPR: {tpr_best:.4f}")
                    #print("_" * 60)
                    accuracy_dict[clique_id].append(correct/total * 100)
                    tpr_dict[clique_id].append(tpr * 100)
                    tnr_dict[clique_id].append(tnr * 100)
                    break
        time.sleep(10)

Processing 0...


  2%|▏         | 2/100 [00:09<07:34,  4.64s/it]


KeyboardInterrupt: 

In [11]:
with open("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_detection/eval_result/810755807/accuracy_dict.pkl", 'wb') as f:
    pickle.dump(accuracy_dict, f)

with open("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_detection/eval_result/810755807/tpr_dict.pkl", 'wb') as f:
    pickle.dump(tpr_dict, f)

with open("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_detection/eval_result/810755807/tnr_dict.pkl", 'wb') as f:
    pickle.dump(tnr_dict, f)